In [1]:
!pip install interpret -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 88.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.9/264.9 kB 13.1 MB/s eta 0:00:00


In [32]:
import pandas as pd
import numpy as np
import sklearn
import shap
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from interpret import show
from interpret.blackbox import ShapKernel

np.random.seed(42)

# **Load data**

In [3]:
df_train = pd.read_csv('/content/drive/MyDrive/AIO_Warm Up_REVIEW/AIO_The Liems/CONQ023/CONQ23_project/data/processed/df_train.csv')
df_test = pd.read_csv('/content/drive/MyDrive/AIO_Warm Up_REVIEW/AIO_The Liems/CONQ023/CONQ23_project/data/processed/df_test.csv')

In [4]:
df_train.head()

,Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,...,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,TenureRatio,IncomePerAge,PromotionLag,Attrition
0,47.0,2,2,2.0,4.0,1,2.0,0,4.0,4.0,...,2.0,3.0,3.0,2.0,1.0,2.0,0.1017,339.83,0.0,0
1,22.0,2,1,2.0,1.0,5,3.0,1,3.0,1.0,...,2.0,3.0,2.0,1.0,2.0,1.0,0.5000,114.68,1.0,0
2,46.0,2,2,3.0,1.0,2,1.0,1,3.0,4.0,...,3.0,3.0,12.0,9.0,4.0,9.0,0.5000,379.67,0.0,0
3,25.0,2,2,13.0,1.0,3,2.0,1,3.0,1.0,...,1.0,3.0,7.0,4.0,0.0,6.0,0.8750,83.84,0.0,0
4,43.0,1,1,9.0,5.0,3,4.0,1,3.0,2.0,...,3.0,3.0,8.0,7.0,4.0,7.0,0.7273,132.07,0.0,0


In [10]:
X_train = df_train.drop('Attrition', axis=1)
y_train = df_train.Attrition
X_test = df_test.drop('Attrition', axis=1)
y_test = df_test.Attrition

# **A. Huấn luyện mô hình LogisticRegression**

In [26]:
# 1. Khởi tạo mô hình
black_box_lr = LogisticRegression(penalty="l2", C=0.1, max_iter=1000, random_state=42)

# 2. Huấn luyện mô hình với tập train
black_box_lr.fit(X_train, y_train)

# 3. Dự đoán và đánh giá trên tập test
y_pred = black_box_lr.predict(X_test)
print("Báo cáo kết quả mô hình Logistic Regression:")
print(classification_report(y_test, y_pred))

Báo cáo kết quả mô hình Logistic Regression:
              precision    recall  f1-score   support

           0       0.92      0.81      0.86       247
           1       0.38      0.62      0.47        47

    accuracy                           0.78       294
   macro avg       0.65      0.71      0.67       294
weighted avg       0.83      0.78      0.80       294



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning:

lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



## Giải thích mô hình Logistic Regression bằng SHAP

In [ ]:
# Khởi tạo ShapKernel từ interpret.blackbox
shap_interpreter_lr = ShapKernel(model=black_box_lr, data=X_train)

In [27]:
# Giải thích cục bộ cho 5 mẫu đầu tiên từ tập kiểm tra
# Pass the model predictions for 'y' when using explain_local if the model is a classifier
shap_explanation_lr = shap_interpreter_lr.explain_local(X_test[:5], y_test[:5])

  0%|          | 0/5 [00:00<?, ?it/s]

In [28]:
# Hiển thị kết quả giải thích
show(shap_explanation_lr)

# **B. Huấn luyện  mô hình Random Forest**

In [25]:
# 1. Khởi tạo mô hình Random Forest
black_box_rf = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Huấn luyện mô hình
black_box_rf.fit(X_train, y_train)

# 3. Dự đoán và đánh giá
y_pred_rf = black_box_rf.predict(X_test)
print("Báo cáo kết quả mô hình Random Forest:")
print(classification_report(y_test, y_pred_rf))

Báo cáo kết quả mô hình Random Forest:
              precision    recall  f1-score   support

           0       0.87      0.94      0.90       247
           1       0.44      0.26      0.32        47

    accuracy                           0.83       294
   macro avg       0.66      0.60      0.61       294
weighted avg       0.80      0.83      0.81       294



## Giải thích mô hình Random Forest bằng SHAP

In [29]:
shap_interpreter_rf = ShapKernel(model=black_box_rf, data=X_train)

# Giải thích cục bộ cho 5 mẫu đầu tiên từ tập kiểm tra
shap_explanation_rf = shap_interpreter_rf.explain_local(X_test[:5], y_test[:5])

  0%|          | 0/5 [00:00<?, ?it/s]

In [30]:
# Hiển thị kết quả giải thích
show(shap_explanation_rf)